# Preparación de datos para análisis en SQL

La preparación de datos es un paso crucial antes de realizar cualquier análisis en SQL. Aquí se mostrará cómo limpiar, transformar y organizar tus datos para que estén listos para consultas eficientes y precisas. Concretaremos, estudiaremos lo siguiente:

* **Limpieza de datos**: Identificación y corrección de errores, manejo de valores nulos y eliminación de duplicados.
* **Detección de valores atípicos (outliers)**: Identificación y manejo de outliers que pueden afectar el análisis.
* **Ingeniería de características**: Creación de nuevas variables a partir de las existentes para mejorar los modelos analíticos.
* **Tratamiento de datos ausentes**: Técnicas para manejar datos faltantes, como imputación o eliminación.

No obstante, una cuestión de interés en tareas de preparación de datos, y gestión de datos en general es la relativa al tratamiento de archivos CSV (Comma-Separated Values). Los archivos CSV son una de las fuentes de datos más comunes en proyectos de análisis de datos. Debido a la ausencia de validaciones de tipo y formato, suelen contener valores ausentes, inconsistencias o errores. Por ello, es fundamental saber importar y preparar correctamente archivos CSV antes de su análisis, garantizando la calidad de los datos y la fiabilidad de las consultas y resultados posteriores. Dedicaremos al final un pequeño apartado al uso de archivos CSV en la preparación de datos.

## Preparación del entorno

Antes de comenzar con la preparación de datos, es importante asegurarse de que el entorno de trabajo esté correctamente configurado. Esto incluye la instalación de las herramientas necesarias, la conexión a la base de datos y la carga de los datos iniciales.

Comenzamos instalando las librería de `jupysql` para conectar Jupyter con bases de datos SQL y `pymysql` para conectar con bases de datos MySQL.

In [ ]:
pip install jupysql pymysql

A continuación, configuramos la conexión a la base de datos MySQL desde Jupyter. Esto nos permitirá ejecutar consultas SQL directamente desde el cuaderno Jupyter. Las credenciales de conexión, como el nombre de usuario, la contraseña, el host y el nombre de la base de datos, deben ser especificadas correctamente para establecer la conexión.

In [ ]:
%load_ext sql
%sql mysql+pymysql://<<your-user>>:<<your-password>>@<<your-host>>:<<your-port>>/<<your-database>> 

## Limpieza de datos

La limpieza de datos es el proceso de identificar y corregir errores en los datos para asegurar su calidad. Esto incluye:

- **Identificación de errores**: Revisar los datos en busca de inconsistencias, errores tipográficos y formatos incorrectos. Para ello, SQL ofrece funciones como `ISNULL()`, `COALESCE()`, y `TRY_CAST()` que pueden ayudar a identificar y manejar estos problemas (En MySQL, `TRY_CAST` se puede implementar mediante `CASE WHEN`).
- **Manejo de valores nulos**: Decidir cómo tratar los valores faltantes, ya sea eliminándolos o imputándolos con valores apropiados.
- **Eliminación de duplicados**: Identificar y eliminar registros duplicados que puedan sesgar los resultados del análisis.
- **Corrección de formatos**: Asegurar que todos los datos estén en el formato correcto (por ejemplo, fechas, números).
- **Validación de rangos**: Verificar que los valores numéricos estén dentro de rangos esperados.

Ejemplo. Obtener id, nombre, sueldo y complemento de empleados sin complemento asignado.

In [ ]:
%%sql
SELECT
    id,
    nombre,
    sueldo,
    complemento
FROM Empleados
WHERE complemento IS NULL;

Ejemplo. Obtener id, nombre y empleo de empleados sin jefe asignado.

In [ ]:
%%sql
SELECT
    id,
    nombre,
    empleo
FROM Empleados
WHERE jefe IS NULL;

Reescribible con `ISNULL()` como

In [ ]:
%%sql
SELECT
    id,
    nombre,
    empleo
FROM Empleados
WHERE ISNULL(jefe);

Ejemplo. Obtener departamentos sin empleados.

In [ ]:
%%sql
SELECT
    d.id,
    d.nombre
FROM Departamentos d
LEFT JOIN Empleados e
    ON d.id = e.departamento_id
WHERE e.id IS NULL;

Ejemplo. Obtener empleados con su sueldo total (sueldo + complemento), manejando valores nulos en complemento. Los valores nulos en complemento se deben considerar como 0.

In [ ]:
%%sql
SELECT
    id,
    nombre,
    sueldo,
    COALESCE(complemento, 0) AS complemento,
    sueldo + COALESCE(complemento, 0) AS salario_total
FROM Empleados;

Ejemplo. Detección de empleados con el mismo nombre, empleo y jefe asignado (posible duplicado).

In [ ]:
%%sql
SELECT nombre, empleo, jefe, COUNT(*) AS cantidad
FROM Empleados
GROUP BY nombre, empleo, jefe
HAVING COUNT(*) > 1;

## Detección de valores atípicos (outliers)

Los valores atípicos (outliers) son datos que se encuentran significativamente alejados del resto de los datos. Estos pueden afectar negativamente el análisis y los resultados. En SQL, puedes identificar outliers utilizando técnicas como:
- **Desviación estándar**: Calcular la media y la desviación estándar de una columna numérica, y considerar como outliers aquellos valores que se encuentran a más de 2 o 3 desviaciones estándar de la media.
- **Cuartiles**: Utilizar los cuartiles para identificar valores que se encuentran fuera del rango intercuartílico (IQR). Los valores que están por debajo de Q1 - 1.5*IQR o por encima de Q3 + 1.5*IQR pueden considerarse outliers.

Ejemplo. Detección de empleados con sueldo por encima o por debajo de dos desviaciones estándar de la media. (Usar la función STDDEV_POP para obtener la desviación estándar poblacional).

In [ ]:
%%sql
SELECT id, nombre, sueldo
FROM Empleados
WHERE sueldo > (SELECT AVG(sueldo) + 2*STDDEV_POP(sueldo) FROM Empleados)
   OR sueldo < (SELECT AVG(sueldo) - 2*STDDEV_POP(sueldo) FROM Empleados);

Ejemplo. Realizar una consulta análoga para detectar empleados con complementos atípicos usando la misma lógica de desviación estándar.

In [ ]:
%%sql
SELECT id, nombre, complemento
FROM Empleados
WHERE complemento IS NOT NULL
  AND (complemento > (SELECT AVG(complemento) + 2*STDDEV_POP(complemento) FROM Empleados WHERE complemento IS NOT NULL)
       OR complemento < (SELECT AVG(complemento) - 2*STDDEV_POP(complemento) FROM Empleados WHERE complemento IS NOT NULL));


## Ingeniería de características

La ingeniería de características implica la creación de nuevas variables a partir de las existentes para mejorar los modelos analíticos. En SQL, esto puede incluir, entre otras:
- **Creación de nuevas columnas**: Utilizar expresiones SQL para derivar nuevas columnas basadas en cálculos o transformaciones de las columnas existentes.
- **Agrupación de datos**: Crear variables categóricas a partir de variables numéricas mediante la agrupación en rangos o categorías.
- **Combinación de columnas**: Concatenar o combinar múltiples columnas para formar nuevas características.


Ejemplo. Obtener id, nombre, sueldo y complemento de cada empleado añadiéndole una columna de sueldo total (sueldo + complemento), manejando valores nulos en complemento. Los valores nulos en complemento se deben considerar como 0.

In [ ]:
%%sql
SELECT 
    id,
    nombre,
    sueldo,
    complemento,
    sueldo + COALESCE(complemento,0) AS sueldo_total
FROM Empleados;

La función `TIMESTAMPDIFF()` puede ser útil para crear nuevas características basadas en diferencias de tiempo entre fechas. Esta función calcula la diferencia entre dos marcas de tiempo en unidades especificadas (como días, meses o años), lo que puede ser útil para analizar la antigüedad de empleados, duración de proyectos u otros aspectos temporales en los datos

Ejemplo. Calcular la antigüedad de cada empleado en años a partir de su fecha de contratación.

In [ ]:
%%sql
SELECT 
    id,
    nombre,
    fecha_entrada,
    TIMESTAMPDIFF(YEAR, fecha_entrada, CURDATE()) AS antiguedad_anios
FROM Empleados;

Otra opción interesante es usar la función `CASE` para crear nuevas características basadas en condiciones específicas. Por ejemplo, podemos categorizar a los emmpleados según su empleo.

Ejemplo. Crear una nueva columna que clasifique a los empleados en `Dirección` y `Operativo` según su empleo. Los empleados con empleo 'Directivo' o 'Director' se clasificarán como 'Dirección', mientras que el resto se clasificará como 'Operativo'.

In [ ]:
%%sql
SELECT 
    id,
    nombre,
    empleo,
    CASE 
        WHEN empleo IN ('Director','Directivo') THEN 'Dirección'
        ELSE 'Operativo'
    END AS Direccion
FROM Empleados;

## Tratamiento de datos ausentes

El tratamiento de datos ausentes es fundamental para asegurar la calidad del análisis. Se trata de valores que no están registrados en el conjunto de datos y aparecen como `NULL` en SQL. Existen varias estrategias para manejar estos datos:
- **Señalización de datos ausentes**: Crear una columna adicional que indique si un valor está ausente o no.
- **Uso de valores predeterminados**: Asignar un valor predeterminado a los campos nulos, como 0 para números o 'Desconocido' para cadenas de texto.
- **Imputación de valores**: Rellenar los valores nulos con valores estadísticos como la media, mediana o moda de la columna.
- **Eliminación de registros**: Si la cantidad de datos ausentes es pequeña, se pueden eliminar los registros que contienen valores nulos.
- **Análisis separado**: Tratar los registros con datos ausentes como un grupo separado en el análisis para entender su impacto.

Ejemplo. Crear una vista que muestre los empleados junto con una columna adicional que indique si el complemento es nulo o no.

In [ ]:
%%sql
CREATE OR REPLACE VIEW empleados_con_indicador_de_complemento AS
SELECT *,
       CASE WHEN complemento IS NULL THEN 1 ELSE 0 END AS complemento_es_nulo
FROM Empleados; 

Ejemplo. Mostrar la vista creada anteriormente.

In [ ]:
%%sql
SELECT *
FROM empleados_con_indicador_de_complemento

Ejemplo. Modificar la vista anterior para que, además del indicador de complemento nulo, muestre el complemento con un valor predeterminado de 0 en lugar de NULL.

In [ ]:
%%sql
CREATE OR REPLACE VIEW empleados_con_indicador_de_complemento AS
SELECT id,
        nombre, empleo, jefe, fecha_entrada, sueldo, COALESCE(complemento, 0) AS complemento, departamento_id,
       CASE WHEN complemento IS NULL THEN 1 ELSE 0 END AS complemento_es_nulo
FROM Empleados;

Ejemplo. Consultar la vista creada.

In [ ]:
%%sql
SELECT * 
FROM empleados_con_indicador_de_complemento;

Ejemplo. Crear una vista que haga una imputación de valores nulos en la columna de complemento, reemplazándolos con la media de los complementos no nulos, y que también incluya un indicador de si el complemento original era nulo.

In [ ]:
%%sql
CREATE OR REPLACE VIEW empleados_con_complemento_imputado_con_media AS
SELECT id,
        nombre, empleo, jefe, fecha_entrada, sueldo, 
       COALESCE(complemento, (SELECT AVG(complemento) FROM Empleados WHERE complemento IS NOT NULL)) AS complemento,
       departamento_id,
       CASE WHEN complemento IS NULL THEN 1 ELSE 0 END AS complemento_es_nulo
FROM Empleados;

Ejemplo. Consultar la vista creada.

In [ ]:
%%sql
SELECT *
FROM empleados_con_complemento_imputado_con_media;

Ejemplo. Eliminar registros de empleados donde el complemento es nulo.

In [ ]:
%%sql
DELETE 
FROM Empleados
WHERE complemento IS NULL;

Ejemplo. Comprobar que los empleados con complemento nulo han sido eliminados.

In [ ]:
%%sql
SELECT *
FROM Empleados;

Ejemplo. Volver a insertar los registros eliminados anteriormente (con complemento nulo) para restaurar la tabla Empleados a su estado original.

In [ ]:
%%sql
INSERT INTO Empleados VALUES
        (7369,'Smith','Ordenanza',7902,'2000-12-17',800,NULL,20);
INSERT INTO Empleados VALUES
        (7566,'Jones','Directivo',7839,'2001-04-02',2975,NULL,20);
INSERT INTO Empleados VALUES
        (7698,'Blake','Directivo',7839,'2001-05-01',2850,NULL,30);
INSERT INTO Empleados VALUES
        (7782,'Clark','Directivo',7839,'2001-06-09',2450,NULL,10);
INSERT INTO Empleados VALUES
        (7788,'Scott','Analista',7566,'2002-12-09',3000,NULL,20);
INSERT INTO Empleados VALUES
        (7839,'King','Director',NULL,'2001-11-17',5000,NULL,10);
INSERT INTO Empleados VALUES
        (7876,'Adams','Ordenanza',7788,'2003-01-12',1100,NULL,20);
INSERT INTO Empleados VALUES
        (7900,'James','Ordenanza',7698,'2001-12-03',950,NULL,30);
INSERT INTO Empleados VALUES
        (7902,'Ford','Analista',7566,'2001-12-03',3000,NULL,20);
INSERT INTO Empleados VALUES
        (7934,'Miller','Ordenanza',7782,'2002-01-23',1300,NULL,10);

## Tratamiento de archivos CSV

Los archivos CSV (Comma-Separated Values) son uno de los formatos más utilizados para el intercambio y almacenamiento de datos debido a su simplicidad y compatibilidad con múltiples herramientas y lenguajes. En proyectos reales de análisis de datos, es muy habitual que la información de partida provenga de ficheros CSV generados por sistemas externos, hojas de cálculo o fuentes abiertas de datos.

Sin embargo, los archivos CSV suelen carecer de validaciones estrictas de tipo, integridad o formato, lo que puede dar lugar a valores ausentes, datos mal tipados, inconsistencias o errores que afectan directamente a la calidad del análisis. Por este motivo, resulta fundamental saber importar, inspeccionar y preparar correctamente este tipo de archivos antes de su explotación analítica.

Dominar el tratamiento de archivos CSV permite al analista detectar problemas de calidad de datos, realizar conversiones seguras de tipos, gestionar valores nulos y estructurar la información de forma adecuada dentro de una base de datos relacional. Esta fase de preparación es clave para garantizar consultas SQL fiables, análisis precisos y resultados reproducibles en proyectos de Ciencia de Datos.

### Carga de archivos CSV en MySQL

La carga de archivos CSV en MySQL es un paso fundamental en la preparación de datos, ya que permite incorporar datos externos a una base de datos relacional para su posterior limpieza y análisis. Este proceso incluye la creación previa de la tabla con los tipos de datos adecuados y el uso de mecanismos de carga como LOAD DATA INFILE o herramientas gráficas (por ejemplo, MySQL Workbench). Durante la carga es importante gestionar correctamente delimitadores, codificación, valores nulos y formatos de fecha para evitar errores y garantizar la calidad de los datos importados.

Ejemplo. Cargar en el cuaderno Jupyter el archivo [`nuevo_personal.csv`](https://gist.githubusercontent.com/ualmtorres/4c842fadc6a7cc98389a837e4a7ca78d/raw/d4f7fff58a2e60a72a384d0813fecc6cd941c85c/nuevo_personal.csv).

In [ ]:
import pandas as pd

df = pd.read_csv("https://gist.githubusercontent.com/ualmtorres/4c842fadc6a7cc98389a837e4a7ca78d/raw/d4f7fff58a2e60a72a384d0813fecc6cd941c85c/nuevo_personal.csv")
df.head()

El código anterior realizar lo siguiente:

1. Importa la librería pandas para trabajar con datos tabulares
2. Lee el archivo CSV "nuevo_personal.csv" y lo carga en un DataFrame
3. Muestra las primeras 5 filas del DataFrame con df.head()

**IMPORTANTE**: Los valores nulos en el archivo CSV se representan como celdas vacías. Al cargar el archivo en el DataFrame, pandas automáticamente interpreta estas celdas vacías como valores NaN (Not a Number), que es la representación estándar de valores nulos en pandas. Esto posteriomente lo trataremos para que sean convertidos en NULL al insertar los datos en la base de datos MySQL.

In [ ]:
import numpy as np
import pymysql

# Conexión al MySQL en Aiven
conn = pymysql.connect(
    host='your-aiven-host.aivencloud.com',
    user='your-username',
    password='your-password',
    database='your-database',
    port=28830  # El puerto de la base de datos Aiven
)
cursor = conn.cursor()

In [ ]:
El código anterior realiza lo siguiente:

1. Importa la librería numpy y la librería pymysql, que permite conectar Python con bases de datos MySQL

3. Establece una conexión a una base de datos MySQL hospedada en Aiven con los siguientes parámetros:
   - host: dirección del servidor MySQL en Aiven
   - user: usuario administrador de la base de datos
   - password: contraseña de autenticación
   - database: nombre de la base de datos a utilizar ('p.e. defaultdb')
   - port: puerto específico de conexión (p.e. 28830)

4. Crea un objeto cursor que permite ejecutar consultas SQL y manipular datos en la base de datos

Esta conexión permite interactuar con la base de datos MySQL desde Python, facilitando operaciones como insertar, actualizar, eliminar o consultar datos

In [ ]:
# Insertar fila a fila convirtiendo NaN a None
for row in df.itertuples(index=False):
    values = tuple(None if (isinstance(x, float) and np.isnan(x)) else x for x in row)
    cursor.execute("""
        INSERT INTO Empleados (id, nombre, empleo, jefe, fecha_entrada, sueldo, complemento, departamento_id)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
    """, values)

conn.commit()
cursor.close()
conn.close()

El código anterior lee del DataFrame (`df`) con los datos importados desde el CSV y vuelca fila a fila esos registros en la tabla Empleados de la base de datos MySQL.

1. Se recorre el DataFrame con for row in `df.itertuples(index=False)`. Se excluye el índice porque `df.itertuples()` incluye el índice como primer elemento de la tupla por defecto (`df.index`). A continuación, viene el resto de columnas en el orden del DataFrame.

2. Para cada fila se construye una tupla `values` reemplazando `NaN` por `None` con la expresión (`None` es el equivalente a `NULL` en Python):
         `tuple(None if (isinstance(x, float) and np.isnan(x)) else x for x in row)` hace que los valores ausentes de pandas se inserten como `NULL` en MySQL.

3. Se ejecuta una sentencia `INSERT` parametrizada usando `cursor.execute(...)` con los valores construidos.

4. Al terminar el bucle se hace `conn.commit()` para que los cambios se guarden en la base de datos, y se cierran el cursor y la conexión.


Ejemplo. Comprobar los datos insertados desde el CSV en la tabla Empleados.

In [ ]:
%%sql
SELECT *
FROM Empleados;

Una vez que los datos se han cargado desde un archivo CSV a la base de datos o al entorno de análisis, es cuando se realizarían las operaciones de preparación de datos descritas anteriormente. Esto se debe a que los archivos CSV suelen contener valores ausentes, inconsistencias, errores de formato o tipos de datos incorrectos que pueden afectar la calidad y fiabilidad del análisis.

Las funciones y técnicas de tratamiento de datos vistas anteriormente, como el manejo de valores nulos (ISNULL, COALESCE), la detección de duplicados, la imputación de datos faltantes, la creación de nuevas columnas derivadas y la validación de datos, permiten:

* Limpiar y normalizar los datos importados.
* Detectar y reemplazar valores ausentes de forma segura.
* Transformar los datos para que cumplan con los tipos y formatos esperados.
* Preparar el dataset para análisis SQL precisos, agregaciones correctas y visualizaciones fiables.

En conjunto, la carga de datos y su posterior tratamiento garantizan que el dataset esté completo, consistente y listo para cualquier consulta, modelado o visualización, reduciendo errores y mejorando la calidad del análisis.